In [1]:
from dotenv import load_dotenv
import os   

In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [14]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

len(docs)

9

In [15]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splitted_doc = splitter.split_documents(docs)

In [16]:
len(splitted_doc)

26

In [18]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_doc,
    embedding=embeddings
)

In [ ]:
# same_record = vector_store.similarity_search("Patient name")
# print(same_record[0].page_content)

Report Status    
Female
27 Years:
:
:
:
Age
Gender
Reported        
P
9/7/2025   4:56:00PM
DR NITIN NAHAR
474764803
Ms. NIKITA  CHUDHARY:
:
:
:
:
Name        
Lab No.    
Ref By 
Collected       
A/c Status Final
10/7/2025  6:31:50PM
:Collected at            :Processed at             BHOPAL CC-82
Mr Rachel V John Pata So Vitus John Mig 26 
Graund,Indrapuri, Phone: 8770817968
 
LPL-NATIONAL REFERENCE LAB
National Reference laboratory, Block E, 
Sector 18, Rohini, New Delhi -110085
Test Report      
Test Name Results Units Bio. Ref. Interval
 ------------------------------------------------------------
Dr Beena Chandrasekhar
PhD, Life Sciences
Technical Director - Flowcytometry              
NRL - Dr Lal PathLabs Ltd
Dr. Nagarjun Sai Jaine
MD, Pathology
Fellowship in Hematopathology (AIIMS) 
Consultant- Hemato-Oncopathology & 
Flowcytometry                                     
NRL - Dr Lal PathLabs Ltd
Dr Sunanda
MD, Pathology
Sr. Consultant Pathologist -


In [27]:
## Agent =  Tools | LLM | Prompt

@tool
def retriveal_tools(query:str):
    """
    this tool can help u retive the relevant document based on the query, this document have details about the patient, and the medical report, so u can ask about any details about the patient and the medical report, and this tool will retive the relevant document for u.
    """

    print("Tool called ", query)
    docs = vector_store.similarity_search(query,k=4)
    context = ""
    for doc in docs:
        context += doc.page_content + "\n\n"
    return context

In [28]:
llm = ChatOpenAI(model="gpt-5")


In [29]:
system_prompt = "you are a medical assistant, you have access to the patient's medical report, and the patient's details, you can answer any question about the patient and the medical report, but you have to use the retriveal tool to retive the relevant document for u, and then u can answer the question based on the document that u retive, if u don't know the answer to the question, u can say that u don't know, but u can't make up an answer. Before u answer u look urself are u giving me relevant information or not, if u think that the information that u have is not relevant to the question, u can say that u don't know, but u can't make up an answer."

In [33]:
agent = create_agent(
    model=llm,
    tools=[retriveal_tools],
    system_prompt=system_prompt
)

query = "what is the patient's name? and also what is the date of the medical report? doctor name? and what is the diagnosis for the patient?"

response = agent.invoke({'messages':[{"role":"user","content":query}]})

Tool called  Retrieve the patient record and the most recent medical report including: patient full name, report date, attending/authoring doctor name, and primary diagnosis.


In [34]:
result = response['messages'][-1].content
print(result)

- Patient’s name: Ms. Nikita Chudhary
- Date of the medical report: 09/07/2025, 04:56 PM (as listed under “Reported”)
- Doctor name: Dr. Nitin Nahar (Referring physician on the report)
- Diagnosis: Not specified in the report; this lab report lists test results but does not state a diagnosis.
